# 阿里 PPU：DolphinDB 分钟数据全量内存训练

本 Notebook 从 DolphinDB 按交易日并发读取分钟数据，直接写入 PPU 节点 RAM。训练开始后不再访问 DolphinDB，也不生成分钟 MemMap 文件。700GB 节点默认预留 100GB，基础分钟数组上限设为 550GB。

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('请先进入 AlphaMining-GFlowNet-AlphaEval 仓库根目录')
print('PROJECT_ROOT =', PROJECT_ROOT)

## 安装依赖

安装完成后如 Jupyter 提示重启 Kernel，请重启后从下一单元格继续。

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -r requirements-ddb.txt

## 检查 DDB 环境变量

请在启动 JupyterLab 前设置这些环境变量。Notebook 只检查是否存在，不会打印账号或密码。

In [ ]:
RAM_CACHE_DIR = Path(os.environ.get(
    'ALPHAMINING_RAM_CACHE_DIR', 'results/minute_ppu_ddb_ram/ram_cache'
)).resolve()
os.environ['ALPHAMINING_RAM_CACHE_DIR'] = str(RAM_CACHE_DIR)
cache_manifest = RAM_CACHE_DIR / 'ram_cache_manifest.json'
cache_ready = (cache_manifest.exists() and
               json.loads(cache_manifest.read_text(encoding='utf-8')).get('complete') is True)
source_env = ['DDB_DATABASE', 'DDB_TABLE', 'DDB_TRADE_DAYS_DATABASE']
connection_env = ['DDB_HOST', 'DDB_PORT', 'DDB_USER', 'DDB_PASSWORD']
required_env = source_env + ([] if cache_ready else connection_env)
missing = [name for name in required_env if not os.environ.get(name)]
if missing:
    raise EnvironmentError('缺少环境变量: ' + ', '.join(missing))
print('RAM_CACHE_DIR =', RAM_CACHE_DIR)
print('RAM缓存存在   =', cache_ready)
print('环境变量检查通过；敏感值未显示')

## 检查内存与配置

正式加载前确认可用内存接近 700GB，并把配置中的 `prices_are_adjusted` 改为真实状态。

In [ ]:
import os
import yaml

CONFIG_PATH = Path('configs/minute/ppu_ddb_ram.yaml')
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
total_ram = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
available_ram = os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
print('total RAM GB     =', round(total_ram / 1024**3, 1))
print('available RAM GB =', round(available_ram / 1024**3, 1))
print('load_mode        =', config['dataset']['dolphindb']['load_mode'])
print('build_workers    =', config['dataset']['memory']['build_workers'])
print('reward_workers   =', config['dataset']['memory']['workers'])
print('ram cache        =', config['dataset']['memory'].get('ram_cache_enabled', True))
assert config['dataset']['memory']['reward_parallel_backend'] == 'threading'

## 选择运行阶段

首次完整运行保留分钟主流程开关为 `True`。如果 GFlowNet 已经训练完成并且 `alpha_pool.csv`、`alpha_factor_matrix.csv.gz` 已生成，将 `RUN_GFLOWNET_TRAINING=False`，直接执行后处理。指数增强需要先把本地生成的三指数标签上传到 PPU，再单独开启两个 `RUN_INDEX_*` 开关。

In [ ]:
RUN_GFLOWNET_TRAINING = True
RUN_ALPHA_EVAL = True
RUN_LIGHTGBM = True
PACKAGE_RESULTS = True
REBUILD_DAILY_PRICE_IF_MISSING = True

# 指数增强默认关闭，避免首次分钟训练因尚未上传指数标签而中断。
RUN_INDEX_ALPHA_EVAL = False
RUN_INDEX_LIGHTGBM = False
INDEX_MODEL_EXPERIMENTS = 'lambdarank'  # 多个实验用逗号分隔
INDEX_BACKTEST_EXPERIMENT = 'lambdarank'  # 本地回测使用其中一个实验

## 开始加载并训练

启动时先检查普通 NumPy 磁盘快照。命中会显示 `[DDBRAMCache] hit` 并跳过DDB连接；未命中才显示 `[DDBRAM]` 从DDB加载，完成后保存快照。无论哪种路径，训练期间都只使用普通RAM数组，不使用MemMap。

In [ ]:
import subprocess
import sys

if RUN_GFLOWNET_TRAINING:
    subprocess.run([
        sys.executable, 'scripts/train_cpu.py',
        '--mode', 'minute', '--config', str(CONFIG_PATH),
    ], check=True)
else:
    print('跳过 GFlowNet 训练，使用已有分钟因子产物')

## 验收分钟训练产物并转换格式

训练结束时已经加载最佳 checkpoint、生成 Alpha Pool，并把分钟表达式日内聚合为日频因子。这里检查日期、重复键和覆盖率，再把压缩 CSV 转成 AlphaEval/LightGBM 使用的 Pickle。

In [ ]:
import pandas as pd
import shutil

outputs = config['outputs']
checkpoint_path = Path(outputs['checkpoint'])
pool_path = Path(outputs['alpha_pool'])
factor_csv_path = Path(outputs['factor_matrix'])
factor_pickle_path = Path(outputs['factor_matrix_pickle'])
daily_price_path = Path(outputs['daily_price'])
legacy_daily_price_path = RAM_CACHE_DIR / 'daily_price.pkl'
if not daily_price_path.exists() and legacy_daily_price_path.exists():
    daily_price_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(legacy_daily_price_path, daily_price_path)
    print('从旧RAM快照迁移日频文件:', daily_price_path)
if not daily_price_path.exists() and REBUILD_DAILY_PRICE_IF_MISSING:
    missing_ddb = [name for name in connection_env + source_env if not os.environ.get(name)]
    if missing_ddb:
        raise EnvironmentError('补建daily_price.pkl需要DDB环境变量: ' + ', '.join(missing_ddb))
    print('缺少独立日频文件；只从DDB聚合日频数据，不重训GFlowNet...')
    subprocess.run([
        sys.executable, 'scripts/export_ddb_daily.py',
        '--config', str(CONFIG_PATH),
    ], check=True)
required = [checkpoint_path, pool_path, factor_csv_path, daily_price_path]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('训练产物不完整: ' + ', '.join(missing))

factors = pd.read_csv(factor_csv_path)
factors['date'] = pd.to_datetime(factors['date']).dt.normalize()
factors['code'] = factors['code'].astype(str)
if factors.duplicated(['date', 'code']).any():
    raise ValueError('因子矩阵存在重复 date/code')
factor_columns = [column for column in factors if column not in ('date', 'code')]
if not factor_columns:
    raise ValueError('因子矩阵没有分钟因子列')
coverage = factors[factor_columns].notna().mean().sort_values()
min_coverage = float(config['reward']['min_coverage'])
low_coverage = coverage[coverage < min_coverage]
if len(low_coverage):
    raise ValueError(f'存在低覆盖因子(<{min_coverage:.0%}): {low_coverage.to_dict()}')
factor_pickle_path.parent.mkdir(parents=True, exist_ok=True)
factors.to_pickle(factor_pickle_path)
print('rows =', len(factors))
print('dates =', factors['date'].nunique())
print('stocks =', factors['code'].nunique())
print('date range =', factors['date'].min().date(), factors['date'].max().date())
print('factors =', len(factor_columns))
print('coverage min/median/max =', round(coverage.min(), 4), round(coverage.median(), 4), round(coverage.max(), 4))
print('pickle saved =', factor_pickle_path)

## 运行 AlphaEval

AlphaEval 只使用 2020–2023 样本内区间，计算 RankIC、ICIR、稳定性、扰动鲁棒性和 DPP 多样性。

In [ ]:
alpha_eval_path = Path(outputs['alpha_eval'])
if RUN_ALPHA_EVAL:
    subprocess.run([
        sys.executable, '-m', 'src.alpha_eval.run_evaluation',
        '--config', str(CONFIG_PATH),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--metadata', str(pool_path),
        '--output', str(alpha_eval_path),
    ], check=True)
elif not alpha_eval_path.exists():
    raise FileNotFoundError(f'跳过 AlphaEval，但结果不存在: {alpha_eval_path}')
evaluation = pd.read_csv(alpha_eval_path)
selected = evaluation.loc[evaluation['dpp_selected'].astype(bool), 'factor']
if selected.empty:
    raise ValueError('AlphaEval 没有选中任何因子')
print('evaluated =', len(evaluation), 'selected =', len(selected))
display(evaluation.head(30))

## 运行 LightGBM

使用 AlphaEval 入选因子，带 5 日 purge 做滚动训练，并只输出 2024–2026 的股票预测分数。

In [ ]:
lightgbm_dir = Path(outputs['lightgbm_dir'])
prediction_path = lightgbm_dir / 'prediction_score.csv'
if RUN_LIGHTGBM:
    subprocess.run([
        sys.executable, 'scripts/audit_lightgbm_inputs.py',
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--evaluation', str(alpha_eval_path),
        '--horizon', str(config['dataset']['horizon']),
        '--output', str(lightgbm_dir / 'input_audit.csv'),
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'src.model.run_lightgbm',
        '--config', str(CONFIG_PATH),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--evaluation', str(alpha_eval_path),
        '--output-dir', str(lightgbm_dir),
    ], check=True)
elif not prediction_path.exists():
    raise FileNotFoundError(f'跳过 LightGBM，但预测文件不存在: {prediction_path}')
prediction = pd.read_csv(prediction_path)
prediction['signal_date'] = pd.to_datetime(prediction['signal_date'])
print('prediction rows =', len(prediction))
print('prediction dates =', prediction['signal_date'].nunique())
print('prediction range =', prediction['signal_date'].min().date(), prediction['signal_date'].max().date())
display(pd.read_csv(lightgbm_dir / 'model_metrics.csv').tail(20))

## 分钟因子指数增强（PPU）

本阶段不重新训练 GFlowNet。请先在本地使用 `configs/index_enhancement/default.yaml` 生成严格的历史成分、指数权重和 `t+5/t+1` 超额收益标签，再把整个 `results/index_enhancement_labels/` 目录上传到 PPU 仓库根目录。

开启 `RUN_INDEX_ALPHA_EVAL` 后，沪深300、中证500和中证1000分别在 2020–2023 样本内进行 AlphaEval/DPP；开启 `RUN_INDEX_LIGHTGBM` 后，分别执行带 5 日 purge 的滚动模型。PPU 只训练和打包模型，不运行 RQAlphaPlus。

In [ ]:
INDEX_KEYS = ['csi300', 'csi500', 'csi1000']
INDEX_LABEL_ROOT = Path('results/index_enhancement_labels')
INDEX_ALPHA_EVAL_ROOT = Path('results/minute_index_alpha_eval')
INDEX_MODEL_ROOT = Path('results/minute_index_model_experiments')
INDEX_EXPERIMENT_CONFIG = Path('configs/index_enhancement/model_experiments.yaml')
index_experiments = [value.strip() for value in INDEX_MODEL_EXPERIMENTS.split(',') if value.strip()]
if not index_experiments:
    raise ValueError('INDEX_MODEL_EXPERIMENTS 不能为空')
if RUN_INDEX_LIGHTGBM and INDEX_BACKTEST_EXPERIMENT not in index_experiments:
    raise ValueError('INDEX_BACKTEST_EXPERIMENT 必须包含在 INDEX_MODEL_EXPERIMENTS 中')

index_artifact_paths = []
index_summary = {}
if RUN_INDEX_ALPHA_EVAL or RUN_INDEX_LIGHTGBM:
    label_paths = [INDEX_LABEL_ROOT / key / 'labels.pkl' for key in INDEX_KEYS]
    missing_labels = [str(path) for path in label_paths if not path.exists()]
    if missing_labels:
        raise FileNotFoundError(
            '缺少指数标签。请先在本地生成并上传 results/index_enhancement_labels/: ' +
            ', '.join(missing_labels)
        )

if RUN_INDEX_ALPHA_EVAL:
    subprocess.run([
        sys.executable, '-m', 'src.alpha_eval.run_index_evaluation',
        '--config', str(CONFIG_PATH),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--metadata', str(pool_path),
        '--labels', str(INDEX_LABEL_ROOT),
        '--output', str(INDEX_ALPHA_EVAL_ROOT),
        '--indexes', ','.join(INDEX_KEYS),
        '--target-column', 'target_excess_return',
    ], check=True)
elif RUN_INDEX_LIGHTGBM:
    missing_evaluation = [
        str(INDEX_ALPHA_EVAL_ROOT / key / 'alpha_eval_result.csv')
        for key in INDEX_KEYS
        if not (INDEX_ALPHA_EVAL_ROOT / key / 'alpha_eval_result.csv').exists()
    ]
    if missing_evaluation:
        raise FileNotFoundError(
            '跳过指数AlphaEval，但历史结果不存在: ' + ', '.join(missing_evaluation)
        )

if RUN_INDEX_LIGHTGBM:
    subprocess.run([
        sys.executable, '-m', 'src.index_enhancement.model_experiments',
        '--training-config', str(CONFIG_PATH),
        '--experiment-config', str(INDEX_EXPERIMENT_CONFIG),
        '--price', str(daily_price_path),
        '--factors', str(factor_pickle_path),
        '--evaluation', str(alpha_eval_path),
        '--labels', str(INDEX_LABEL_ROOT),
        '--index-alpha-eval', str(INDEX_ALPHA_EVAL_ROOT),
        '--output', str(INDEX_MODEL_ROOT),
        '--indexes', ','.join(INDEX_KEYS),
        '--experiments', ','.join(index_experiments),
    ], check=True)

if RUN_INDEX_ALPHA_EVAL or RUN_INDEX_LIGHTGBM:
    for index_key in INDEX_KEYS:
        evaluation_path = INDEX_ALPHA_EVAL_ROOT / index_key / 'alpha_eval_result.csv'
        selected_path = INDEX_ALPHA_EVAL_ROOT / index_key / 'selected_factors.csv'
        index_artifact_paths.extend([evaluation_path, selected_path])
        evaluation_frame = pd.read_csv(evaluation_path)
        index_summary[index_key] = {
            'evaluated_factors': int(len(evaluation_frame)),
            'selected_factors': int(evaluation_frame['dpp_selected'].astype(bool).sum()),
            'experiments': {},
        }
        if RUN_INDEX_LIGHTGBM:
            for experiment in index_experiments:
                model_dir = INDEX_MODEL_ROOT / experiment / index_key
                model_artifacts = [
                    model_dir / 'prediction_score.csv',
                    model_dir / 'model_metrics.csv',
                    model_dir / 'feature_importance.csv',
                    model_dir / 'lgbm_model.joblib',
                ]
                index_artifact_paths.extend(model_artifacts)
                score = pd.read_csv(model_artifacts[0])
                index_summary[index_key]['experiments'][experiment] = {
                    'prediction_rows': int(len(score)),
                    'prediction_dates': int(pd.to_datetime(score['signal_date']).nunique()),
                    'prediction_file': str(model_artifacts[0]),
                }
    missing_index_artifacts = [str(path) for path in index_artifact_paths if not path.exists()]
    if missing_index_artifacts:
        raise FileNotFoundError('指数增强产物不完整: ' + ', '.join(missing_index_artifacts))
    if RUN_INDEX_LIGHTGBM:
        index_backtest_source = Path('configs/index_enhancement/default.yaml')
        index_backtest_config = yaml.safe_load(index_backtest_source.read_text(encoding='utf-8'))
        for index_key in INDEX_KEYS:
            index_backtest_config['indexes'][index_key]['prediction_dir'] = str(
                INDEX_MODEL_ROOT / INDEX_BACKTEST_EXPERIMENT / index_key
            )
        index_backtest_config['backtest']['base_config'] = str(CONFIG_PATH)
        index_backtest_config['backtest']['output_root'] = (
            'results/minute_index_enhancement_backtest'
        )
        index_backtest_config['backtest']['portfolio_mode'] = 'benchmark_optimized'
        index_backtest_config_path = Path('results/minute_index_backtest.yaml')
        index_backtest_config_path.write_text(
            yaml.safe_dump(index_backtest_config, allow_unicode=True, sort_keys=False),
            encoding='utf-8',
        )
        index_artifact_paths.append(index_backtest_config_path)
        index_summary['backtest_config'] = str(index_backtest_config_path)
    print(json.dumps(index_summary, ensure_ascii=False, indent=2))
else:
    print('跳过指数增强；如需运行，请上传三指数标签并开启 RUN_INDEX_ALPHA_EVAL/RUN_INDEX_LIGHTGBM')

## 保存后处理清单并打包

压缩包只保存继续研究和本地回测必需的模型、表达式、评价结果、预测与指标；不会打包数十 GB 的 RAM 分钟快照。

In [ ]:
from datetime import datetime, timezone
import zipfile

artifact_paths = [
    checkpoint_path, pool_path, alpha_eval_path, prediction_path,
    Path(outputs['metrics']), Path(outputs['trajectory_metrics']),
    lightgbm_dir / 'model_metrics.csv',
    lightgbm_dir / 'feature_importance.csv',
    lightgbm_dir / 'lgbm_model.joblib',
]
artifact_paths.extend(index_artifact_paths)
missing = [str(path) for path in artifact_paths if not path.exists()]
if missing:
    raise FileNotFoundError('后处理产物不完整: ' + ', '.join(missing))
manifest = {
    'pipeline': 'minute_ppu_postprocess',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_period': [config['dataset']['mining_start_date'], config['dataset']['mining_end_date']],
    'prediction_period': [config['lightgbm']['prediction_start_date'], config['lightgbm']['prediction_end_date']],
    'factor_rows': len(factors),
    'factors': len(factor_columns),
    'selected_factors': len(selected),
    'prediction_rows': len(prediction),
    'index_enhancement': index_summary,
    'artifacts': [str(path) for path in artifact_paths],
}
manifest_path = Path(outputs['postprocess_manifest'])
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
package_path = Path(outputs['artifact_package'])
if PACKAGE_RESULTS:
    package_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(CONFIG_PATH, CONFIG_PATH.as_posix())
        archive.write(manifest_path, manifest_path.as_posix())
        for path in artifact_paths:
            archive.write(path, path.as_posix())
    print('package =', package_path, 'MB =', round(package_path.stat().st_size / 1024**2, 1))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 下载到本地回测

下载并在本地仓库根目录解压 `results/minute_ppu_ddb_ram/minute_ppu_artifacts.zip`。指数增强结果开启时，压缩包同时包含三套预测和自动生成的 `results/minute_index_backtest.yaml`。确认本地存在 `data/index_weights.csv.gz` 后运行：

```bash
/opt/miniconda3/envs/rqsdk/bin/python -m rqalpha_strategy.run_index_enhancement \
  --config results/minute_index_backtest.yaml
```

该配置固定使用 `benchmark_optimized`，并读取 `INDEX_BACKTEST_EXPERIMENT` 指定的三套预测。RQAlphaPlus 不在 PPU 上运行、不使用代理，也不需要下载 RAM 快照或完整因子矩阵。